# PensComplaintProject Analysis

This notebook reads the pension complaints dataset (`pension_complaints_dataset.csv`), performs basic data cleaning and generates pivot tables and charts used in the project report.  

## Objectives

- Load the complaints dataset
- Clean key fields (e.g., standardise root cause names, remove invalid policy numbers)
- Generate summary statistics and pivot tables
- Visualise complaint root causes and cross-team interactions

*Note:* To run this notebook, place `pension_complaints_dataset.csv` in the same directory as this notebook.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display all columns when printing DataFrames
pd.set_option('display.max_columns', None)


In [ ]:

# Load the dataset
file_path = 'pension_complaints_dataset.csv'

data = pd.read_csv(file_path)
print(f"Dataset loaded: {data.shape[0]} rows and {data.shape[1]} columns")

data.head()


In [ ]:

# Basic data cleaning
# Clean Policy_Number: ensure last 6 digits are numeric and mark invalid entries

def validate_policy(num):
    s = str(num)
    return s[-6:].isdigit()

# Apply validation and drop invalid records
valid_mask = data['Policy_Number'].apply(validate_policy)
invalid_count = (~valid_mask).sum()
print(f"Dropping {invalid_count} rows with invalid policy numbers")

data = data[valid_mask].copy()

# Standardise root cause names (strip whitespace and title-case)
data['Root_Cause'] = data['Root_Cause'].str.strip().str.title()

# Fill missing handling team values with 'Unknown'
data['Handling_Team'] = data['Handling_Team'].fillna('Unknown')

# Display cleaned dataset summary
data.info()


In [ ]:

# Pivot table: count of complaints by root cause
root_counts = data['Root_Cause'].value_counts().reset_index()
root_counts.columns = ['Root_Cause', 'Count']

plt.figure(figsize=(8, 4))
sns.barplot(x='Root_Cause', y='Count', data=root_counts, palette='viridis')
plt.title('Number of Complaints by Root Cause')
plt.ylabel('Number of Complaints')
plt.xlabel('Root Cause')
plt.xticks(rotation=45)
plt.show()

root_counts


In [ ]:

# Pivot table: complaints count by Responsible Team and Root Cause
team_pivot = pd.pivot_table(data, index='Responsible_Team', columns='Root_Cause', values='Complaint_ID', aggfunc='count', fill_value=0)

# Heatmap visualisation
plt.figure(figsize=(10, 6))
sns.heatmap(team_pivot, annot=True, fmt='d', cmap='YlGnBu')
plt.title('Complaints by Responsible Team and Root Cause')
plt.ylabel('Responsible Team')
plt.xlabel('Root Cause')
plt.show()

team_pivot
